# Atlas Forge Engine 0.6 API — T4 GPU

Versão corrigida para o Dead-zone. Usa T4 GPU, TripoSR, FastAPI e Cloudflare Tunnel.

A remoção de fundo é o comportamento padrão do TripoSR. O argumento `--no-remove-bg` só é enviado quando a opção estiver desmarcada no app.


## 1. Confirmar GPU T4


In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Sessão sem GPU. Selecione T4 GPU no tipo de ambiente de execução.')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True,check=True)
gpu_info = gpu.stdout.strip()
print('GPU disponível:', gpu_info)
if 'T4' not in gpu_info.upper():
    raise RuntimeError(f'GPU atual não é T4: {gpu_info}')
print('✅ T4 confirmada.')


## 2. Preparar o motor isolado


In [ ]:
import pathlib, shutil, subprocess, sys
ROOT=pathlib.Path('/content/TripoSR')
ENV=pathlib.Path('/content/atlas_forge_env')
JOBS_ROOT=pathlib.Path('/content/atlas_jobs')
JOBS_ROOT.mkdir(parents=True,exist_ok=True)
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(ROOT)],check=True)
if ENV.exists(): shutil.rmtree(ENV)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','virtualenv'],check=True)
subprocess.run([sys.executable,'-m','virtualenv','--system-site-packages',str(ENV)],check=True)
PYTHON=str(ENV/'bin'/'python')
subprocess.run([PYTHON,'-m','pip','install','-q','--upgrade','pip','setuptools','wheel'],check=True)
subprocess.run([PYTHON,'-m','pip','install','-q','--ignore-installed','numpy==1.26.4','cupy-cuda12x==13.3.0','onnxruntime==1.20.1','rembg==2.0.61','omegaconf==2.3.0','Pillow==10.1.0','einops==0.7.0','transformers==4.35.0','trimesh==4.0.5','huggingface-hub==0.17.3','imageio[ffmpeg]','xatlas==0.0.9','moderngl==5.10.0','pygltflib','fastapi','uvicorn[standard]','python-multipart'],check=True)
subprocess.run([PYTHON,'-m','pip','install','--no-build-isolation','git+https://github.com/tatsy/torchmcubes.git@3aef8afa5f21b113afc4f4ea148baee850cbd472'],check=True)
print('✅ Motor preparado.')


## 3. Criar a API corrigida


In [ ]:
from pathlib import Path
APP_FILE=Path('/content/atlas_forge_api.py')
APP_FILE.write_text(r'''
import json, os, shutil, subprocess, threading, uuid
from pathlib import Path
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
ROOT=Path('/content/TripoSR')
PYTHON='/content/atlas_forge_env/bin/python'
JOBS_ROOT=Path('/content/atlas_jobs')
JOBS_ROOT.mkdir(parents=True,exist_ok=True)
app=FastAPI(title='Atlas Forge Engine API',version='0.6')
app.add_middleware(CORSMiddleware,allow_origins=['*'],allow_credentials=False,allow_methods=['*'],allow_headers=['*'])
def write_status(d,p): (d/'status.json').write_text(json.dumps(p,ensure_ascii=False,indent=2),encoding='utf-8')
def read_status(job):
    p=JOBS_ROOT/job/'status.json'
    if not p.exists(): raise HTTPException(status_code=404,detail='job não encontrado')
    return json.loads(p.read_text(encoding='utf-8'))
def patch_bake_file():
    p=ROOT/'tsr'/'bake_texture.py'
    c=p.read_text(encoding='utf-8')
    c=c.replace('moderngl.create_context(standalone=True)',"moderngl.create_context(standalone=True, backend='egl')")
    c=c.replace('positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1])','positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1], device=scene_code.device)')
    c=c.replace('rgb_f = queried_grid["color"].numpy().reshape(-1, 3)','rgb_f = queried_grid["color"].detach().cpu().numpy().reshape(-1, 3)')
    p.write_text(c,encoding='utf-8')
def convert_to_glb(d,out):
    target=d/'model.glb'
    meshes=list(out.rglob('*.obj'))+list(out.rglob('*.glb'))
    if not meshes: raise RuntimeError('Nenhuma malha encontrada.')
    src=meshes[0]
    if src.suffix.lower()=='.glb': shutil.copy2(src,target)
    else:
        code=f"import trimesh; s=trimesh.load(r'{src}',force='scene',process=False); s.export(r'{target}',file_type='glb')"
        r=subprocess.run([PYTHON,'-c',code],capture_output=True,text=True)
        if r.returncode!=0: raise RuntimeError(r.stderr)
    return target
def process_job(job,input_path,quality,remove_bg):
    d=JOBS_ROOT/job; out=d/'output'
    try:
        write_status(d,{'jobId':job,'status':'processing','message':'Gerando modelo 3D...','progress':20,'modelUrl':None})
        patch_bake_file()
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True,exist_ok=True)
        tex={'fast':'512','balanced':'1024','high':'2048'}.get(quality,'1024')
        cmd=[PYTHON,'run.py',input_path,'--output-dir',str(out),'--bake-texture','--texture-resolution',tex]
        if not remove_bg: cmd.append('--no-remove-bg')
        env=os.environ.copy(); env['PYOPENGL_PLATFORM']='egl'
        r=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True,env=env)
        if r.returncode!=0: raise RuntimeError((r.stderr or 'Falha no TripoSR').strip())
        write_status(d,{'jobId':job,'status':'processing','message':'Convertendo para GLB...','progress':80,'modelUrl':None})
        convert_to_glb(d,out)
        u=f'/models/{job}'
        write_status(d,{'jobId':job,'status':'done','message':'Modelo pronto.','progress':100,'modelUrl':u,'glbUrl':u,'url':u})
    except Exception as e:
        msg=str(e)
        if 'unrecognized arguments' in msg: msg='O motor recebeu uma opção incompatível. Reabra o notebook 0.6 e execute novamente as etapas 3, 4 e 5.'
        write_status(d,{'jobId':job,'status':'error','message':msg,'progress':100,'modelUrl':None})
@app.get('/')
def root(): return {'ok':True,'service':'atlas-forge-engine-api','version':'0.6'}
@app.post('/image-to-3d')
async def image_to_3d(image:UploadFile=File(...),quality:str=Form('balanced'),removeBackground:str=Form('true'),prompt:str=Form('')):
    suf=Path(image.filename or 'upload.png').suffix.lower()
    if suf not in {'.png','.jpg','.jpeg','.webp'}: raise HTTPException(status_code=400,detail='Use PNG, JPG ou WEBP.')
    job=uuid.uuid4().hex[:12]; d=JOBS_ROOT/job; d.mkdir(parents=True,exist_ok=True); inp=d/f'input{suf}'; inp.write_bytes(await image.read())
    write_status(d,{'jobId':job,'status':'queued','message':'Job criado.','progress':5,'modelUrl':None})
    threading.Thread(target=process_job,args=(job,str(inp),quality,str(removeBackground).lower()!='false'),daemon=True).start()
    return {'jobId':job,'status':'queued','statusUrl':f'/image-to-3d-status/{job}','pollUrl':f'/image-to-3d-status/{job}'}
@app.get('/image-to-3d-status/{job_id}')
def status(job_id:str): return read_status(job_id)
@app.get('/models/{job_id}')
def model(job_id:str):
    p=JOBS_ROOT/job_id/'model.glb'
    if not p.exists(): raise HTTPException(status_code=404,detail='Modelo ainda não disponível.')
    return FileResponse(p,media_type='model/gltf-binary',filename=f'{job_id}.glb')
''',encoding='utf-8')
print('✅ API 0.6 criada.')


## 4. Reiniciar o servidor


In [ ]:
import os, signal, subprocess, time
subprocess.run(['pkill','-f','uvicorn.*atlas_forge_api'],check=False)
time.sleep(2)
server=subprocess.Popen([PYTHON,'-m','uvicorn','atlas_forge_api:app','--host','0.0.0.0','--port','8000'],cwd='/content')
time.sleep(5)
print('✅ Servidor 0.6 iniciado. PID:',server.pid)


## 5. Publicar e conectar ao Dead-zone


In [ ]:
import os, re, subprocess, time, urllib.request
from IPython.display import display, HTML
cf='/content/cloudflared'
if not os.path.exists(cf):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cf)
    os.chmod(cf,0o755)
subprocess.run(['pkill','-f','cloudflared tunnel'],check=False)
log='/content/cloudflared.log'
with open(log,'w') as f:
    tunnel=subprocess.Popen([cf,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=f,stderr=subprocess.STDOUT)
public_url=None
for _ in range(40):
    time.sleep(1)
    text=open(log,errors='ignore').read() if os.path.exists(log) else ''
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if m: public_url=m.group(0); break
if not public_url: raise RuntimeError('Não foi possível gerar a URL pública.')
deadzone='https://dead-zone-steel.vercel.app/?atlas_engine='+urllib.parse.quote(public_url,safe='')
display(HTML(f'''<a href="{deadzone}" target="_blank" style="display:block;padding:16px;background:#aaff00;color:#0b0f0c;text-align:center;font-weight:800;border-radius:10px;text-decoration:none">CONECTAR AO DEAD-ZONE</a><p>{public_url}</p>'''))
print('✅ Motor 0.6 publicado:',public_url)
